# Null-Text Inversion + Prompt-to-Prompt con ControlNet

Questo notebook combina:
1. **DDIM Inversion** (w=1) con ControlNet per ottenere il latente rumoroso z_T
2. **Null-Text Optimization** per ottimizzare gli embedding non-condizionati per ogni step
3. **Ricostruzione fedele** dell'immagine originale usando i null-text ottimizzati
4. **Prompt-to-Prompt (P2P)** con iniezione delle mappe di cross-attention dalla ricostruzione alla generazione con un nuovo prompt

## Parte 0 — Setup e Installazione

In [ ]:
!pip install controlnet-aux

In [ ]:
# === CONFIGURAZIONE ===
CONTROLNET_SCALE = 0.5        # Peso del ControlNet
NUM_STEPS = 50                 # Step DDIM (inversion + generazione)
GUIDANCE_SCALE = 7.5           # CFG scale per null-text e generazione
NULL_TEXT_LR = 0.07            # Learning rate per l'ottimizzazione null-text
NULL_TEXT_INNER_STEPS = 25     # Iterazioni interne dell'ottimizzazione
TAU = 0.8                      # Soglia P2P: inietta mappe per il primo tau% degli step
IMAGE_SIZE = 512               # Risoluzione immagine
DEVICE = "cuda"

## Parte 1 — Caricamento Modello e Immagine

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DDIMScheduler

dtype = torch.float16

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=dtype
).to(DEVICE)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype
).to(DEVICE)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

print(f"Scheduler config: steps_offset={pipe.scheduler.config.steps_offset}, "
      f"set_alpha_to_one={pipe.scheduler.config.set_alpha_to_one}")

In [ ]:
# === PROMPT ORIGINALE ===
# Deve corrispondere all'immagine di input
source_prompt = "A man with wearing a tight white shirt"

# === PROMPT TARGET PER P2P ===
target_prompt = "A man with wearing a tight white shirt"

In [ ]:
# === CARICAMENTO E PREPROCESSING IMMAGINE ===
def load_image(path, size=IMAGE_SIZE):
    """Carica un'immagine, la ridimensiona e la converte in tensore [-1, 1]"""
    image_pil = Image.open(path).convert("RGB").resize((size, size))
    image_np = np.array(image_pil).astype(np.float32) / 255.0
    image_np = image_np[None].transpose(0, 3, 1, 2)  # [1, 3, H, W]
    image_tensor = torch.from_numpy(image_np).to(DEVICE, dtype=dtype)
    image_tensor = 2 * image_tensor - 1  # [0,1] -> [-1,1]
    return image_tensor, image_pil

# MODIFICA QUESTO PERCORSO con la tua immagine
IMAGE_PATH = "/kaggle/input/datasets/nightshield/pose333/pose1.png"
image_tensor, image_pil = load_image(IMAGE_PATH)
# seconda img
IMAGE_PATH2 = "/kaggle/input/datasets/nightshield/pose333/pose2.png"
image_tensor2, image_pil2 = load_image(IMAGE_PATH2)

plt.imshow(image_pil)
plt.title("Immagine Originale")
plt.axis("off")
plt.show()

In [ ]:
# === ESTRAZIONE CONTROL IMAGE (Canny) ===
import cv2

CANNY_LOW = 100   # Soglia inferiore Canny
CANNY_HIGH = 200  # Soglia superiore Canny

def extract_control_image(image_pil, low=CANNY_LOW, high=CANNY_HIGH):
    """Estrae la mappa dei bordi (Canny) e la converte in tensore [1, 3, H, W] in [0, 1]"""
    image_np = np.array(image_pil)
    edges = cv2.Canny(image_np, low, high)            # [H, W] uint8
    edges = np.stack([edges, edges, edges], axis=2)   # [H, W, 3]
    canny_pil = Image.fromarray(edges)
    canny_np = edges.astype(np.float32) / 255.0
    canny_np = canny_np.transpose(2, 0, 1)            # HWC -> CHW
    canny_tensor = torch.from_numpy(canny_np).unsqueeze(0).to(DEVICE, dtype=dtype)
    return canny_tensor, canny_pil

control_image, control_pil = extract_control_image(image_pil)
control_image2, control_pil2 = extract_control_image(image_pil2)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image_pil); ax[0].set_title("Originale"); ax[0].axis("off")
ax[1].imshow(control_pil); ax[1].set_title("Bordi Canny"); ax[1].axis("off")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image_pil2); ax[0].set_title("Originale"); ax[0].axis("off")
ax[1].imshow(control_pil2); ax[1].set_title("Bordi Canny"); ax[1].axis("off")
plt.show()

In [ ]:
# === ENCODING NELLO SPAZIO LATENTE ===
with torch.no_grad():
    z0 = pipe.vae.encode(image_tensor).latent_dist.mean * 0.18215
print(f"z0 shape: {z0.shape}")  # [1, 4, 64, 64]

## Parte 2 — DDIM Inversion (w=1)

In [ ]:
from diffusers import DDIMInverseScheduler

def ddim_inversion_w1_controlnet(
    pipe, z_0, prompt, control_image,
    num_steps=NUM_STEPS, device=DEVICE,
    controlnet_conditioning_scale=CONTROLNET_SCALE
):
    """
    DDIM Inversion con w=1 (solo condizionato, senza CFG).
    Ritorna la traiettoria [z_0, z_1, ..., z_T] (T+1 elementi).
    """
    inverse_scheduler = DDIMInverseScheduler.from_config(pipe.scheduler.config)
    inverse_scheduler.set_timesteps(num_steps, device=device)

    # Embedding condizionato (con w=1 non serve l'uncondizionato)
    text_inputs = pipe.tokenizer(
        prompt, padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        return_tensors="pt"
    )
    text_embeddings = pipe.text_encoder(text_inputs.input_ids.to(device))[0]

    z_t = z_0.clone().to(device)
    trajectory = [z_t]

    pipe.unet.eval()
    pipe.controlnet.eval()

    with torch.no_grad():
        for t in inverse_scheduler.timesteps:
            down_res, mid_res = pipe.controlnet(
                z_t, t,
                encoder_hidden_states=text_embeddings,
                controlnet_cond=control_image,
                conditioning_scale=controlnet_conditioning_scale,
                return_dict=False
            )
            noise_pred = pipe.unet(
                z_t, t,
                encoder_hidden_states=text_embeddings,
                down_block_additional_residuals=down_res,
                mid_block_additional_residual=mid_res
            ).sample

            z_t = inverse_scheduler.step(noise_pred, t, z_t).prev_sample
            trajectory.append(z_t)

    return trajectory

In [ ]:
trajectory = ddim_inversion_w1_controlnet(
    pipe, z0, source_prompt, control_image
)
print(f"Traiettoria: {len(trajectory)} elementi (z_0 ... z_T)")
print(f"z_T shape: {trajectory[-1].shape}")

## Parte 3 — Null-Text Optimization

In [ ]:
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

def null_text_optimization(
    pipe, prompt, control_image, trajectory,
    lr=NULL_TEXT_LR, num_inner_steps=NULL_TEXT_INNER_STEPS,
    epsilon=1e-5, guidance_scale=GUIDANCE_SCALE,
    controlnet_conditioning_scale=CONTROLNET_SCALE, device=DEVICE
):
    """
    Ottimizza gli embedding non-condizionati (null-text) per ogni timestep.
    Ritorna una lista di tensori [emb_step_0, ..., emb_step_{T-1}].
    """
    scaler = GradScaler('cuda')

    # Embedding condizionato
    text_inputs = pipe.tokenizer(
        prompt, padding="max_length",
        max_length=pipe.tokenizer.model_max_length, return_tensors="pt"
    )
    text_embeddings = pipe.text_encoder(text_inputs.input_ids.to(device))[0].detach()

    # Embedding null-text iniziale
    uncond_inputs = pipe.tokenizer(
        "", padding="max_length",
        max_length=pipe.tokenizer.model_max_length, return_tensors="pt"
    )
    uncond_emb_init = pipe.text_encoder(uncond_inputs.input_ids.to(device))[0].detach()

    pipe.scheduler.set_timesteps(len(trajectory) - 1, device=device)
    optimized_uncond = []

    # Partiamo da z_T e avanziamo sequenzialmente
    z_bar_t = trajectory[-1].clone().detach()

    pbar = tqdm(pipe.scheduler.timesteps, desc="Null-Text Optimization")

    for i, t in enumerate(pbar):
        # Target: il punto precedente nella traiettoria originale
        target_z_prev = trajectory[-(i + 2)].clone().detach()

        # Passaggio CONDIZIONATO (fisso, calcolato una volta)
        with torch.no_grad():
            with autocast('cuda', dtype=torch.float16):
                down_cond, mid_cond = pipe.controlnet(
                    z_bar_t, t,
                    encoder_hidden_states=text_embeddings,
                    controlnet_cond=control_image,
                    conditioning_scale=controlnet_conditioning_scale,
                    return_dict=False
                )
                noise_pred_cond = pipe.unet(
                    z_bar_t, t,
                    encoder_hidden_states=text_embeddings,
                    down_block_additional_residuals=down_cond,
                    mid_block_additional_residual=mid_cond
                ).sample

        # Preparazione ottimizzatore
        if i == 0:
            uncond_emb_t = uncond_emb_init.clone().detach().float()
        else:
            uncond_emb_t = optimized_uncond[-1].clone().detach().float()
        uncond_emb_t.requires_grad_(True)
        optimizer = Adam([uncond_emb_t], lr=lr)

        loss_val = 0.0
        steps_used = 0

        for step in range(num_inner_steps):
            steps_used = step + 1
            optimizer.zero_grad()

            with autocast('cuda', dtype=torch.float16):
                down_uncond, mid_uncond = pipe.controlnet(
                    z_bar_t, t,
                    encoder_hidden_states=uncond_emb_t,
                    controlnet_cond=control_image,
                    conditioning_scale=controlnet_conditioning_scale,
                    return_dict=False
                )
                noise_pred_uncond = pipe.unet(
                    z_bar_t, t,
                    encoder_hidden_states=uncond_emb_t,
                    down_block_additional_residuals=down_uncond,
                    mid_block_additional_residual=mid_uncond
                ).sample

                noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
                z_prev_pred = pipe.scheduler.step(noise_pred, t, z_bar_t).prev_sample

            loss = F.mse_loss(z_prev_pred.float(), target_z_prev.float())
            loss_val = loss.item()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if loss_val < epsilon:
                break

        pbar.set_postfix({"inner": steps_used, "loss": f"{loss_val:.6f}"})

        optimized_uncond.append(uncond_emb_t.clone().detach().to(pipe.unet.dtype))

        # Aggiornamento sequenziale di z_bar_t
        with torch.no_grad():
            with autocast('cuda', dtype=torch.float16):
                down_uncond, mid_uncond = pipe.controlnet(
                    z_bar_t, t,
                    encoder_hidden_states=optimized_uncond[-1],
                    controlnet_cond=control_image,
                    conditioning_scale=controlnet_conditioning_scale,
                    return_dict=False
                )
                noise_pred_uncond = pipe.unet(
                    z_bar_t, t,
                    encoder_hidden_states=optimized_uncond[-1],
                    down_block_additional_residuals=down_uncond,
                    mid_block_additional_residual=mid_uncond
                ).sample

                noise_pred_cfg = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
                z_bar_t = pipe.scheduler.step(noise_pred_cfg, t, z_bar_t).prev_sample

    return optimized_uncond

In [ ]:
optimized_null_texts = null_text_optimization(
    pipe, source_prompt, control_image, trajectory
)
print(f"Null-text ottimizzati: {len(optimized_null_texts)} step, shape: {optimized_null_texts[0].shape}")

## Parte 4 — P2P Attention Processor

Creiamo un `AttnProcessor` custom per diffusers che:
- In modalita **STORE**: salva le mappe di cross-attention alla risoluzione 16x16
- In modalita **INJECT**: sovrascrive le mappe con quelle salvate (fino a soglia tau)

In [ ]:
class P2PController:
    """
    Controller centralizzato per Prompt-to-Prompt.
    Gestisce lo stato (store/inject) condiviso da tutti i processor.
    """
    def __init__(self, expected_shape=None):
        # expected_shape: mantenuto solo per retro-compatibilita / log.
        # L'iniezione ora avviene a TUTTE le risoluzioni (vedi filtro nel processor).
        self.expected_shape = expected_shape or (IMAGE_SIZE // 32) ** 2
        self.store_mode = False
        self.inject_mode = False
        self.source_maps = []     # Mappe salvate durante la generazione source
        self.inject_maps = []     # Mappe da iniettare (con None dopo tau)
        self.inject_index = 0     # Contatore per l'iniezione sequenziale
        # Ramo attivo: "cond" o "uncond". P2P agisce SOLO sul ramo condizionato
        # (come nel paper / codice ufficiale, che edita solo attn[h//2:]).
        self.cur_branch = "cond"

    def start_store(self):
        self.source_maps.clear()
        self.store_mode = True
        self.inject_mode = False

    def stop_store(self):
        self.store_mode = False

    def prepare_inject(self, num_steps, tau):
        """
        Prepara le mappe per l'iniezione con soglia tau.
        Le prime tau*num_steps step iniettano le mappe source,
        le restanti lasciano il modello libero (None).
        """
        total_maps = len(self.source_maps)
        if total_maps == 0:
            print("ATTENZIONE: nessuna mappa salvata!")
            return
        maps_per_step = total_maps // num_steps
        inject_limit = int(tau * num_steps) * maps_per_step
        self.inject_maps = self.source_maps[:inject_limit] + [None] * (total_maps - inject_limit)
        self.inject_index = 0
        self.inject_mode = True
        self.store_mode = False
        print(f"P2P: {total_maps} mappe totali, {maps_per_step} per step, "
              f"iniezione per i primi {int(tau * num_steps)}/{num_steps} step")

    def stop_inject(self):
        self.inject_mode = False
        self.inject_index = 0


class P2PAttnProcessor:
    """
    AttnProcessor custom che salva/inietta le mappe di cross-attention.
    Replica il comportamento di CrossAttention.forward() in ldm/modules/attention.py.
    """
    def __init__(self, controller: P2PController):
        self.controller = controller

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, *args, **kwargs):
        is_cross = encoder_hidden_states is not None

        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim
        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )
        attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        query = attn.head_to_batch_dim(query)
        key = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)

        attention_probs = attn.get_attention_scores(query, key, attention_mask)

        # === LOGICA P2P ===
        # Filtro: solo cross-attention (77 token CLIP), a TUTTE le risoluzioni
        # (64x64, 32x32, 16x16, mid) come nel codice ufficiale P2P, e SOLO sul
        # ramo condizionato (cur_branch == "cond"), come fa il paper.
        if (is_cross
            and attention_probs.shape[-1] == 77
            and self.controller.cur_branch == "cond"):

            # STORE: salva le mappe dalla generazione source
            if self.controller.store_mode:
                self.controller.source_maps.append(attention_probs.detach().half().cpu())

            # INJECT: sovrascrive le mappe nella generazione target
            if self.controller.inject_mode:
                if self.controller.inject_index < len(self.controller.inject_maps):
                    injected = self.controller.inject_maps[self.controller.inject_index]
                    if injected is not None:
                        attention_probs = injected.to(
                            device=attention_probs.device, dtype=attention_probs.dtype
                        )
                    self.controller.inject_index += 1
        # === FINE LOGICA P2P ===

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)

        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states

In [ ]:
# === REGISTRAZIONE DEL PROCESSOR SULLA UNET ===
controller = P2PController()

processors = {}
for key in pipe.unet.attn_processors.keys():
    processors[key] = P2PAttnProcessor(controller)
pipe.unet.set_attn_processor(processors)

print(f"P2P processor registrato su {len(processors)} layer di attenzione")
print(f"Expected shape per P2P: {controller.expected_shape}")

## Parte 5 — Ricostruzione SOURCE + Salvataggio Mappe P2P

Generiamo con i null-text ottimizzati e il prompt originale.
Simultaneamente, salviamo le mappe di cross-attention per P2P.

In [ ]:
def generate_with_null_text(
    pipe, prompt, optimized_uncond, control_image, trajectory,
    guidance_scale=GUIDANCE_SCALE,
    controlnet_conditioning_scale=CONTROLNET_SCALE,
    device=DEVICE
):
    """
    Generazione con null-text ottimizzati.
    Se il P2P controller e' in store_mode, le mappe vengono salvate automaticamente.
    Lo store/inject P2P avviene SOLO sul ramo condizionato: impostiamo
    controller.cur_branch prima di ciascuna delle due passate UNet.
    """
    text_inputs = pipe.tokenizer(
        prompt, padding="max_length",
        max_length=pipe.tokenizer.model_max_length, return_tensors="pt"
    )
    text_embeddings = pipe.text_encoder(text_inputs.input_ids.to(device))[0].detach()

    pipe.scheduler.set_timesteps(len(trajectory) - 1, device=device)
    z_t = trajectory[-1].clone().detach()

    with torch.no_grad():
        for i, t in enumerate(tqdm(pipe.scheduler.timesteps, desc="Generazione")):
            uncond_emb = optimized_uncond[i].detach()

            # --- RAMO CONDIZIONATO --- (P2P attivo: store/inject)
            controller.cur_branch = "cond"
            with autocast('cuda', dtype=torch.float16):
                down_cond, mid_cond = pipe.controlnet(
                    z_t, t,
                    encoder_hidden_states=text_embeddings,
                    controlnet_cond=control_image,
                    conditioning_scale=controlnet_conditioning_scale,
                    return_dict=False
                )
                noise_pred_cond = pipe.unet(
                    z_t, t,
                    encoder_hidden_states=text_embeddings,
                    down_block_additional_residuals=down_cond,
                    mid_block_additional_residual=mid_cond
                ).sample

            # --- RAMO NON-CONDIZIONATO --- (P2P inattivo)
            controller.cur_branch = "uncond"
            with autocast('cuda', dtype=torch.float16):
                down_uncond, mid_uncond = pipe.controlnet(
                    z_t, t,
                    encoder_hidden_states=uncond_emb,
                    controlnet_cond=control_image,
                    conditioning_scale=controlnet_conditioning_scale,
                    return_dict=False
                )
                noise_pred_uncond = pipe.unet(
                    z_t, t,
                    encoder_hidden_states=uncond_emb,
                    down_block_additional_residuals=down_uncond,
                    mid_block_additional_residual=mid_uncond
                ).sample

            # CFG
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
            z_t = pipe.scheduler.step(noise_pred, t, z_t).prev_sample

    # Ripristina lo stato di default del controller
    controller.cur_branch = "cond"
    return z_t.detach()


def latent_to_image(pipe, latent):
    """Decodifica un latente in immagine numpy uint8."""
    with torch.no_grad():
        image = pipe.vae.decode(latent / 0.18215).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
        return (image * 255).astype(np.uint8)

In [ ]:
# === GENERAZIONE SOURCE CON SALVATAGGIO MAPPE ===
controller.start_store()

source_latent = generate_with_null_text(
    pipe, source_prompt, optimized_null_texts, control_image, trajectory
)

controller.stop_store()
source_image = latent_to_image(pipe, source_latent)

print(f"Mappe di attenzione salvate: {len(controller.source_maps)}")

# Confronto: originale vs ricostruzione
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(image_pil); ax[0].set_title("Originale"); ax[0].axis("off")
ax[1].imshow(source_image); ax[1].set_title("Ricostruzione (Null-Text)"); ax[1].axis("off")
plt.suptitle(f"Prompt: '{source_prompt}'")
plt.tight_layout()
plt.show()

## Confronto PSNR: Null-Text + ControlNet vs Null-Text + SD1.5 puro

Carichiamo SD1.5 senza ControlNet su `cuda:1`, eseguiamo la stessa pipeline
(DDIM inversion w=1 → Null-Text Optimization → ricostruzione) e confrontiamo
la qualità di ricostruzione tramite PSNR.

In [ ]:
# === CARICAMENTO SD1.5 VANILLA SU cuda:1 ===
from diffusers import StableDiffusionPipeline

DEVICE_V = "cuda:1"

pipe_v = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=dtype
).to(DEVICE_V)
pipe_v.scheduler = DDIMScheduler.from_config(pipe_v.scheduler.config)

# Encoding dell'immagine sullo stesso device
image_tensor_v = image_tensor.to(DEVICE_V)
with torch.no_grad():
    z0_v = pipe_v.vae.encode(image_tensor_v).latent_dist.mean * 0.18215
print(f"SD1.5 vanilla caricato su {DEVICE_V}, z0_v shape: {z0_v.shape}")

In [ ]:
# === FUNZIONI VANILLA (senza ControlNet) ===

def ddim_inversion_w1_vanilla(pipe_v, z_0, prompt, num_steps=NUM_STEPS, device=DEVICE_V):
    inverse_scheduler = DDIMInverseScheduler.from_config(pipe_v.scheduler.config)
    inverse_scheduler.set_timesteps(num_steps, device=device)

    text_inputs = pipe_v.tokenizer(
        prompt, padding="max_length",
        max_length=pipe_v.tokenizer.model_max_length, return_tensors="pt"
    )
    text_embeddings = pipe_v.text_encoder(text_inputs.input_ids.to(device))[0]

    z_t = z_0.clone()
    traj = [z_t]

    pipe_v.unet.eval()
    with torch.no_grad():
        for t in inverse_scheduler.timesteps:
            noise_pred = pipe_v.unet(z_t, t, encoder_hidden_states=text_embeddings).sample
            z_t = inverse_scheduler.step(noise_pred, t, z_t).prev_sample
            traj.append(z_t)
    return traj


def null_text_optimization_vanilla(
    pipe_v, prompt, trajectory_v,
    lr=NULL_TEXT_LR, num_inner_steps=NULL_TEXT_INNER_STEPS,
    epsilon=1e-5, guidance_scale=GUIDANCE_SCALE, device=DEVICE_V
):
    scaler = GradScaler(device)

    text_inputs = pipe_v.tokenizer(
        prompt, padding="max_length",
        max_length=pipe_v.tokenizer.model_max_length, return_tensors="pt"
    )
    text_embeddings = pipe_v.text_encoder(text_inputs.input_ids.to(device))[0].detach()

    uncond_inputs = pipe_v.tokenizer(
        "", padding="max_length",
        max_length=pipe_v.tokenizer.model_max_length, return_tensors="pt"
    )
    uncond_emb_init = pipe_v.text_encoder(uncond_inputs.input_ids.to(device))[0].detach()

    pipe_v.scheduler.set_timesteps(len(trajectory_v) - 1, device=device)
    optimized_uncond = []
    z_bar_t = trajectory_v[-1].clone().detach()

    pbar = tqdm(pipe_v.scheduler.timesteps, desc="Null-Text Opt (vanilla)")

    for i, t in enumerate(pbar):
        target_z_prev = trajectory_v[-(i + 2)].clone().detach()

        with torch.no_grad():
            with autocast(device, dtype=torch.float16):
                noise_pred_cond = pipe_v.unet(
                    z_bar_t, t, encoder_hidden_states=text_embeddings
                ).sample

        if i == 0:
            uncond_emb_t = uncond_emb_init.clone().detach().float()
        else:
            uncond_emb_t = optimized_uncond[-1].clone().detach().float()
        uncond_emb_t.requires_grad_(True)
        optimizer = Adam([uncond_emb_t], lr=lr)

        loss_val = 0.0
        steps_used = 0

        for step in range(num_inner_steps):
            steps_used = step + 1
            optimizer.zero_grad()

            with autocast(device, dtype=torch.float16):
                noise_pred_uncond = pipe_v.unet(
                    z_bar_t, t, encoder_hidden_states=uncond_emb_t
                ).sample
                noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
                z_prev_pred = pipe_v.scheduler.step(noise_pred, t, z_bar_t).prev_sample

            loss = F.mse_loss(z_prev_pred.float(), target_z_prev.float())
            loss_val = loss.item()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if loss_val < epsilon:
                break

        pbar.set_postfix({"inner": steps_used, "loss": f"{loss_val:.6f}"})
        optimized_uncond.append(uncond_emb_t.clone().detach().to(pipe_v.unet.dtype))

        with torch.no_grad():
            with autocast(device, dtype=torch.float16):
                noise_pred_uncond = pipe_v.unet(
                    z_bar_t, t, encoder_hidden_states=optimized_uncond[-1]
                ).sample
                noise_pred_cfg = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
                z_bar_t = pipe_v.scheduler.step(noise_pred_cfg, t, z_bar_t).prev_sample

    return optimized_uncond


def generate_with_null_text_vanilla(
    pipe_v, prompt, optimized_uncond, trajectory_v,
    guidance_scale=GUIDANCE_SCALE, device=DEVICE_V
):
    text_inputs = pipe_v.tokenizer(
        prompt, padding="max_length",
        max_length=pipe_v.tokenizer.model_max_length, return_tensors="pt"
    )
    text_embeddings = pipe_v.text_encoder(text_inputs.input_ids.to(device))[0].detach()

    pipe_v.scheduler.set_timesteps(len(trajectory_v) - 1, device=device)
    z_t = trajectory_v[-1].clone().detach()

    with torch.no_grad():
        for i, t in enumerate(tqdm(pipe_v.scheduler.timesteps, desc="Generazione (vanilla)")):
            uncond_emb = optimized_uncond[i].detach()

            with autocast(device, dtype=torch.float16):
                noise_pred_cond = pipe_v.unet(
                    z_t, t, encoder_hidden_states=text_embeddings
                ).sample
                noise_pred_uncond = pipe_v.unet(
                    z_t, t, encoder_hidden_states=uncond_emb
                ).sample

            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)
            z_t = pipe_v.scheduler.step(noise_pred, t, z_t).prev_sample

    return z_t.detach()


print("Funzioni vanilla definite.")

In [ ]:
# === ESECUZIONE PIPELINE VANILLA + CONFRONTO PSNR ===

# 1) DDIM Inversion (vanilla, cuda:1)
trajectory_v = ddim_inversion_w1_vanilla(pipe_v, z0_v, source_prompt)
print(f"Traiettoria vanilla: {len(trajectory_v)} elementi")

# 2) Null-Text Optimization (vanilla, cuda:1)
opt_null_texts_v = null_text_optimization_vanilla(pipe_v, source_prompt, trajectory_v)
print(f"Null-text vanilla ottimizzati: {len(opt_null_texts_v)} step")

# 3) Ricostruzione (vanilla, cuda:1)
source_latent_v = generate_with_null_text_vanilla(
    pipe_v, source_prompt, opt_null_texts_v, trajectory_v
)
source_image_v = latent_to_image(pipe_v, source_latent_v)

# 4) PSNR
original_np = np.array(image_pil.resize((IMAGE_SIZE, IMAGE_SIZE))).astype(np.float64)

def compute_psnr(img1, img2):
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255.0 ** 2 / mse)

psnr_cn = compute_psnr(original_np, source_image.astype(np.float64))
psnr_vanilla = compute_psnr(original_np, source_image_v.astype(np.float64))

print(f"\n{'='*50}")
print(f"PSNR  Null-Text + ControlNet (Canny):  {psnr_cn:.2f} dB")
print(f"PSNR  Null-Text + SD1.5 puro:          {psnr_vanilla:.2f} dB")
print(f"Delta:                                  {psnr_cn - psnr_vanilla:+.2f} dB")
print(f"{'='*50}")

# 5) Plot confronto visivo
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(image_pil)
ax[0].set_title("Originale")
ax[0].axis("off")
ax[1].imshow(source_image)
ax[1].set_title(f"Null-Text + ControlNet\nPSNR: {psnr_cn:.2f} dB")
ax[1].axis("off")
ax[2].imshow(source_image_v)
ax[2].set_title(f"Null-Text + SD1.5 puro\nPSNR: {psnr_vanilla:.2f} dB")
ax[2].axis("off")
plt.suptitle("Confronto ricostruzione: ControlNet vs SD1.5 vanilla", fontsize=14)
plt.tight_layout()
plt.show()

# Libera VRAM su cuda:1
del pipe_v, z0_v, trajectory_v, opt_null_texts_v, source_latent_v, image_tensor_v
torch.cuda.empty_cache()

## Parte 6 — Generazione TARGET con P2P

Generiamo con il prompt modificato, iniettando le mappe di cross-attention
dalla ricostruzione source per i primi `tau * NUM_STEPS` step.

In [ ]:
# === PREPARAZIONE INIEZIONE ===
controller.prepare_inject(num_steps=NUM_STEPS, tau=TAU)

target_prompt = "A girl with wearing a tight white shirt"

# === GENERAZIONE TARGET CON P2P ===
target_latent = generate_with_null_text(
    pipe, target_prompt, optimized_null_texts, control_image, trajectory
)

controller.stop_inject()
target_image = latent_to_image(pipe, target_latent)

# Confronto: originale vs source vs target
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(image_pil)
ax[0].set_title("Originale")
ax[0].axis("off")
ax[1].imshow(source_image)
ax[1].set_title(f"Source: '{source_prompt}'")
ax[1].axis("off")
ax[2].imshow(target_image)
ax[2].set_title(f"Target (P2P, tau={TAU}): '{target_prompt}'")
ax[2].axis("off")
plt.tight_layout()
plt.show()

## Parte 7 — Salvataggio per gradio_swap.py

Salva z_T e null-text ottimizzati per usarli nella repo ControlNet con gradio.

In [ ]:
# Salva su CPU per compatibilita
null_texts_cpu = [t.cpu() for t in optimized_null_texts]
z_T_cpu = trajectory[-1].cpu()

torch.save(null_texts_cpu, 'optimized_null_texts.pt')
torch.save(z_T_cpu, 'trajectory_xT.pt')

print(f"Salvati: optimized_null_texts.pt ({len(null_texts_cpu)} step)")
print(f"Salvati: trajectory_xT.pt (shape {z_T_cpu.shape})")
print("Questi file possono essere usati da gradio_swap.py nella repo ErneAttention.")

In [ ]:
import torch

# 1. Imposta il device (usa la GPU se disponibile, altrimenti resta su CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Carica i file precedentemente salvati
null_texts_loaded = torch.load('optimized_null_texts.pt')
z_T_loaded = torch.load('trajectory_xT.pt')

# 3. Sposta i tensori sul device corretto per l'inferenza
optimized_null_texts = [t.to(device) for t in null_texts_loaded]
z_T = z_T_loaded.to(device)

print(f"Caricati {len(optimized_null_texts)} step di null-text sul device: {device}")
print(f"Caricata trajectory_xT con shape {z_T.shape} sul device: {device}")

## Extra — Esperimenta con tau diversi

In [ ]:
# Prova diversi valori di tau per vedere l'effetto
tau_values = [0.0, 0.4, 0.6, 0.8, 1.0]
results = []

for tau_val in tau_values:
    controller.prepare_inject(num_steps=NUM_STEPS, tau=tau_val)
    lat = generate_with_null_text(
        pipe, target_prompt, optimized_null_texts, control_image, trajectory
    )
    controller.stop_inject()
    results.append(latent_to_image(pipe, lat))
    print(f"tau={tau_val} completato")

fig, axes = plt.subplots(1, len(tau_values) + 1, figsize=(4 * (len(tau_values) + 1), 4))
axes[0].imshow(image_pil)
axes[0].set_title("Originale")
axes[0].axis("off")
for i, (tau_val, img) in enumerate(zip(tau_values, results)):
    axes[i + 1].imshow(img)
    axes[i + 1].set_title(f"tau={tau_val}")
    axes[i + 1].axis("off")
plt.suptitle(f"P2P: '{source_prompt}' → '{target_prompt}'")
plt.tight_layout()
plt.show()